# Check the Table references

Compares every Table reference in `references/` with its table in the warehouse, through your query, and prints what differs, with the line to change. Then it checks the Key of each table that has one: that no two rows share it. It changes nothing in the warehouse; its first cell refreshes the Reference sheet, so it writes `references/reference_sheet.csv` and the Database modules again, and no other file. A Saved table you haven't created yet reports that DESCRIBE failed, as each of the Example projects' does: that is expected. It reads each table from the Reference sheet: its columns, their types, its partition columns and its Key. A partitioned table's Key isn't checked here, since that would read every Partition, and a Saved table's storage, its folder and file format, isn't in the sheet: the last cell checks each on the table's own Table reference, and the check prints the lines to write there.

In [ ]:
import pathlib
import sys
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # the Project: one up from admin/

from sqlglot_composer import project

In [ ]:
import csv

from sqlglot_composer import Table, check_key, check_table_reference
from warehouse import query

# The Reference sheet: a row per column of every Table reference in references/, with its
# type, and "yes" for a partition column and for a column in the Key.
print(project.refresh_reference_sheet())
sheet = pathlib.Path.cwd().parent / "references" / "reference_sheet.csv"
with open(sheet, encoding="utf-8", newline="") as file:
    rows = list(csv.DictReader(file))

In [ ]:
# Each table, as the Reference sheet holds it, compared with the table in the warehouse;
# then its Key, where it has one.
for name in sorted({row["table"] for row in rows}):
    columns = [row for row in rows if row["table"] == name]
    partitions = [row["column"] for row in columns if row["partition"] == "yes"]
    key = [row["column"] for row in columns if row["in_key"] == "yes"]
    table = Table(name, columns={row["column"]: row["type"] or None for row in columns},
                  partitioned_by=partitions, key=key or None)
    found = check_table_reference(table, query=query)
    print(found)
    database, _, short = name.rpartition(".")
    if not key:
        print(f"{name} has no Key, so there is no Key to check.")
    elif not found.ok:
        print(f"{name}: its Key is checked once the problems above are fixed.")
    elif partitions:
        print(f"{name} is partitioned by {' and '.join(partitions)}, so its Key isn't checked "
              "here: that would read every Partition. Check it on some, in the last cell:\n"
              f"    from references.{database}__{short} import {short}\n"
              f"    print(check_key({short}, query=query, "
              f"where=equals({short}.{partitions[0]}, <a value>)))")
    else:
        print(check_key(table, query=query))
    print()

In [ ]:
# A table checked on its own Table reference: a Saved table's storage, its folder and file
# format, which the Reference sheet doesn't hold, and a partitioned table's Key, on the
# Partitions where= picks. Uncomment, and write your table, and a value of its partition
# column, as the cell above printed:
# from sqlglot_composer import equals
# from references.ops__job_runs import job_runs
# print(check_table_reference(job_runs, query=query))
# print(check_key(job_runs, query=query, where=equals(job_runs.dt, "2026-09-24")))